In [1]:
import os
import re
import requests
import wikipediaapi
import numpy as np
import json
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from typing import List, Dict, Any
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from sentence_transformers import SentenceTransformer
from datetime import datetime, timezone

#### Konfiguration

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project) Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
HEADERS = {"User-Agent": USER_AGENT}
DATA_WIKI = Path('../data/wiki')
DATA_JSON = Path('../data/ausflugziele')
URL_SPREEWALDLISTE = Path('../data/web/')
BASE_URL_BOOTSVERLEIH = "https://www.spreewald-info.de"

# Embedding-Modell

EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Milvus-Verbindung
connections.connect("default", host="127.0.0.1", port="19530")


#### 1. Quelle: Wikipedia

In [3]:
wiki = wikipediaapi.Wikipedia(language='de',
                              extract_format=wikipediaapi.ExtractFormat.WIKI,
                              user_agent=USER_AGENT)

# URL's laden
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls

# Wiki texte von unwichtige Teile bereinigen
def remove_references(text: str) -> str:
    stop_sections = ['Einzelnachweise', 'Weblinks', 'Literatur', 'Quellen', 'Fußnoten']
    for section in stop_sections:
        if section in text:
            text = text.split(section)[0]
    return text.strip()

def clean_wiki_text(text):
    text = remove_references(text)
    # Zeilenende normalisieren
    text = text.replace("\r\n", "\n")

    return text.strip()

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}

    text = clean_wiki_text(page.text)
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }

# Text in Chunks teilen
def chunk_wiki_text(text: str, max_words: int = 200, overlap: int = 50):
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)

        start = max(0, end - overlap)  # Überlappung für Kontext
        if start >= len(words):
            break

    return chunks

urls_wiki = load_urls(DATA_WIKI)
wiki_chunks = []

for url in urls_wiki:
    title = extract_title_from_url(url)
    if not title:
        continue

    data = load_wikipedia_text(title)
    if "text" not in data:
        continue

    chunks = chunk_wiki_text(data['text'], max_words=200, overlap=50)
    for i, chunk in enumerate(chunks):
        wiki_chunks.append({
            "text": chunk,
            "title": data['title'],
            "url": data['source'],
            "license": data['license']
        })
print(f"{len(wiki_chunks)} Text-Chunks mit Metadaten erstellt.")

# Collection-Schema definieren
collection_name = "wiki_collection"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)  # alte collection löschen
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="license", dtype=DataType.VARCHAR, max_length=64),
]
schema = CollectionSchema(fields, description="Wikipedia Chunks für semantische Suche")

collection = Collection(name=collection_name, schema=schema)

#Daten in Milvus einfügen
# Milvus generiert IDs automatisch
texts = [chunk['text'] for chunk in wiki_chunks]
titles = [chunk['title'] for chunk in wiki_chunks]
urls = [chunk['url'] for chunk in wiki_chunks]
licenses = [chunk['license'] for chunk in wiki_chunks]

wiki_embeddings = embedding_model.encode(texts).astype(np.float32).tolist()  # Text in Vektoren umwandeln
collection.insert([wiki_embeddings, texts, titles, urls, licenses])

# print(f"Embeddings für {len(wiki_embeddings)} Wiki-Chunks erstellt.")

#Index erstellen (HNSW)
index_params = {
    "index_type": "HNSW",  # Graph-basiertert
    "metric_type": "COSINE",  # Ähnlichkeitsmaß für die Suche
    "params": {"M": 16, "efConstruction": 200}
    # M - Anzahl der Verbindungen (Kanten) pro Punkt. Standart: bessere Suchqualität. efC.. - wie viel Kandidaten schaut den Algorithmus beim Einfügen eines Punktes
}
collection.create_index(field_name="embedding", index_params=index_params)
#print(f"Index für Collection '{collection_name}' erstellt.")

collection.load()


#Semantische Suche in Milvus
def semantic_search_milvus_wiki(query: str, k: int = 5):
    collection = Collection("wiki_collection")
    q_emb = embedding_model.encode([query]).astype(np.float32).tolist()
    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"ef": 50}},
        # ef:50 - Kandidaten werden überprüft, um bessere Ergebnisse zurückzugeben
        limit=k,
        output_fields=["text", "title", "url", "license"]
    )
    output = []
    for result in results:
        for hit in result:
            output.append({
                "score": hit.score,
                "text": hit.entity.get("text"),
                "title": hit.entity.get("title"),
                "url": hit.entity.get("url"),
                "license": hit.entity.get("license")
            })
    return output

# Test
# query = "Geschichte, Spreewald"
# results = semantic_search_milvus_wiki(query, k=5)
#
# for r in results:
#     print(f"\n--- Treffer: {r['title']} ---")
#     print(f"Score: {r['score']:.4f}")
#     print(f"URL: {r['url']}")
#     print(f"Text: {r['text'][:500]}...")  # Nur die ersten 300 Zeichen anzeigen


207 Text-Chunks mit Metadaten erstellt.


#### 2. Eigene JSON-Datei

In [4]:
def load_json_data(path):
    data = []
    for filename in os.listdir(path):
        # Nur JSON-Dateien verarbeiten
        if not filename.endswith('.json'):
            continue
            # Öffnen und Laden der JSON-Datei im Lesemodus
        with open(os.path.join(path, filename), 'r', encoding='utf-8') as f:
                # Inhalt der JSON-Datei laden
            content = json.load(f)
            if isinstance(content, list):
                  data.extend(content)
            else:
                data.append(content)
    return data

In [5]:
#Collection erstellen
data = load_json_data(DATA_JSON)

docs_list = []

for i, doc in enumerate(data, start=1):
    beschreibung = doc.get('beschreibung')
    if not beschreibung:
        continue

    text = f"""
    Name: {doc.get('name')}
    Ort: {doc.get('ort')}
    Region: {doc.get('region')}
    Öffnungszeiten: {doc.get('oeffnungszeiten')}
    Eintrittspreise: {doc.get('eintrittspreise')}
    Zielgruppe: {doc.get('zielgruppe')}
    Content_type: {doc.get('content_type')}
    Beschreibung: {doc.get('beschreibung')}
    """.strip()

    docs_list.append({
        "name": doc.get("name") or "",
        "ort": doc.get("ort") or "",
        "region": doc.get("region") or "",
        "oeffnungszeiten": doc.get("oeffnungszeiten"),
        "text": text

    })

	# Vorschau ausgeben
#             print(f"\n--- Dokument {i}: {doc.get('name', 'Unbekannt')} ---")
#             print(f"\n{doc_entry['text'][:800]}...")  # nur die ersten 300 Zeichen
#
#         else:
#             print(f"Fehlende Beschreibung in Dokument: {doc.get('name', 'unknown')}")
# print(f"{len(docsListe)} Dokumente für RAG vorbereitet.")

collection_name = "ausflug_collection"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=512),
    FieldSchema(name="ort", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="region", dtype=DataType.VARCHAR, max_length=256)
]

schema = CollectionSchema(fields, description="Ausflugziele JSON Datenbank")

collection = Collection(collection_name, schema)

texts = [doc["text"] or "" for doc in docs_list]
names = [doc["name"] or "" for doc in docs_list]
orte = [doc["ort"] or "" for doc in docs_list]
regions = [doc["region"] or "" for doc in docs_list]

embeddings = embedding_model.encode(texts).astype("float32").tolist()

collection.insert([
    embeddings,
    texts,
    names,
    orte,
    regions
])

#Index erstellen
index_params = {

    "index_type": "HNSW",
    "metric_type": "COSINE",  #Ähnlichkeitsmaß für die Suche

    "params": {
        "M": 16,  # Anzahl der Verbindungen pro Knoten
        "efConstruction": 200
    }
}

collection.create_index(
    field_name="embedding",
    index_params=index_params
)

collection.load()

def semantic_search_json(query, k=5):
    collection = Collection("ausflug_collection")

    q_emb = embedding_model.encode([query]).astype("float32").tolist()

    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={
            "metric_type": "COSINE",
            "params": {"ef": 50}
        },
        limit=k,
        output_fields=["text", "name", "ort", "region"]
    )

    output = []

    for hits in results:
        for hit in hits:
            output.append({
                "name": hit.entity.get("name"),
                "ort": hit.entity.get("ort"),
                "region": hit.entity.get("region"),
                "text": hit.entity.get("text"),
                "score": hit.score
            })

    return output


#### 3. Bootsverleih-Webseite (Web Scraping)
Hier werden die Preise für Bootsverleih extrahiert

In [6]:
# Links sammeln: Es gibt mehrere Seiten mit Bootsverleih, die alle unter "/paddeln/bootsverleih/" liegen.
def collect_bootsverleih_links():
    url = BASE_URL_BOOTSVERLEIH + "/paddeln/bootsverleih/"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    links = []

    for a in soup.select("a"):  #alle Links auf der Seite durchgehen
        href = a.get("href")
        if not href:
            continue

        if href.startswith("/paddeln/bootsverleih/") and href != "/paddeln/bootsverleih/":
            links.append(urljoin(BASE_URL_BOOTSVERLEIH, href))
        # if href and "/paddeln/bootsverleih/" in href and href.count("/") >= 3:
        #     links.append(urljoin(BASE, href))

    links = list(set(links))  #Duplikate entfernen
    return links

# links = collect_bootsverleih_links()
# for i, link in enumerate(links, start=1):
#     print(f"{i}. {link}")


# Daten extrahieren
def parse_prices(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] Konnte URL nicht abrufen: {url} – {e}")
        return {"preise": [], "error": str(e)}

    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "Unknown"

    prices = []

    for li in soup.select(".anbieterDetailsInfobox li"):
        text = " ".join(li.stripped_strings)
        if "Euro" in text or "€" in text:
            prices.append(text)

    return {
        "anbieter": title,
        "url": url,
        "preise": prices
    }

# Alle Anbieter und Preise sammeln. parce_prices() ist widerverwendbar
def collect_provider_and_prices():
    links = collect_bootsverleih_links()
    docs = []

    for link in links:
        try:
            data = parse_prices(link)
            docs.append(data)

            # print("\nAnbieter:", data["anbieter"])
            # print("URL:", data["url"])
            # for p in data["preise"]:
            #     print("  -", p)

        except Exception as e:
            print("Fehler bei:", link, e)

    return docs

#JSON für RAG vorbereiten. Die Preise können sich ändern, deswegen werden nicht direckt in den Text stehen
def write_jsonl(docs, path: Path):
    #output_path = Path("providers.jsonl")
    retrieved_at = datetime.now(timezone.utc).isoformat()
    with path.open("w", encoding="utf-8") as f:
        for item in docs:
           #text feld für Embeddings vorbereiten

            anbieter = item.get("anbieter", "")
            source_url = item.get("url", "")

            text = (
                f"Anbieter: {anbieter}\n"
                f"Leistung: Bootsverleih\n"
                f"Quelle: {source_url}\n"
                f"Hinweis: Aktuelle Preise bitte über die Quelle abrufen."
            ).strip()

            doc_entry = {
                "id": source_url,
                "url": source_url,
                "text": text,
                "metadata": {
                    "source_url": source_url,
                    "anbieter": anbieter,
                    "license": "unknown",
                    "retrieved_at": retrieved_at,
                }
            }

            f.write(json.dumps(doc_entry, ensure_ascii=False) + "\n")

docs = collect_provider_and_prices()
write_jsonl(docs, Path("providers.jsonl"))
# embedding
def load_bootsverleih_jsonl(path: Path):
    data_boots = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            data_boots.append(item)
    print(f"{len(data_boots)} Dokumente geladen.")
    return data_boots


data = load_bootsverleih_jsonl(Path("providers.jsonl"))

texts = [item['text'] for item in data]

embeddings_preise = embedding_model.encode(texts)
embeddings = np.array(embeddings_preise).astype('float32')
print(f"Embeddings für {len(embeddings)} Dokumente erstellt.")

# Live Preise abrufen.

def get_bootsverleih_prices(docs: List[dict]):
    results = []

    for d in docs:
        # Preise echtzeitig abrufen

        if not d.get("url"):
            continue
        live_data = parse_prices(d["url"])
        prices = live_data.get("preise", [])
        price_text = "\n".join(f"- {p} " for p in prices) if prices else "Keine Preise gefunden."
        results.append({
            "text": (
                f"Anbieter: {d['anbieter']}\n"
                f"Leistung: Bootsverleih\n"
                f"Preise: {price_text}\n"
                # f"Quelle: {d['url']}\n"
            ),
            "source": d["url"],
            "score": 0.9
        })

    return results

17 Dokumente geladen.
Embeddings für 17 Dokumente erstellt.


#### 5. OpenStreetMap für Öffnungszeiten von Museen.

In [7]:
def search_place(query: str) -> Dict[str, Any]:
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "addressdetails": 1
    }

    r = requests.get(url, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    data = r.json()

    if not data:
        return {}

    return {
        "name": data[0]["display_name"],
        "lat": float(data[0]["lat"]),
        "lon": float(data[0]["lon"]),
        "city": data[0]["address"].get("city") or data[0]["address"].get("town"),
        "state": data[0]["address"].get("state"),
        "country": data[0]["address"].get("country")
    }


def get_museums_opening_hours(lat: float, lon: float, radius: int = 10000):
    query = f"""
    [out:json][timeout:25];

    (
      node["tourism"="museum"](around:{radius},{lat},{lon});
      way["tourism"="museum"](around:{radius},{lat},{lon});
      relation["tourism"="museum"](around:{radius},{lat},{lon});
    );

    out tags center;
    """

    r = requests.post(
        "https://overpass-api.de/api/interpreter",
        data=query,
        headers=HEADERS,
        timeout=60
    )

    r.raise_for_status()
    data = r.json()

    museums = []

    for el in data.get("elements", []):
        tags = el.get("tags", {})

        if not tags.get("name"):
            continue

        lat_val = el.get("lat") or el.get("center", {}).get("lat")
        lon_val = el.get("lon") or el.get("center", {}).get("lon")

        museums.append({
            "name": tags.get("name"),
            "opening_hours": tags.get("opening_hours"),
            "lat": lat_val,
            "lon": lon_val,
            "street": tags.get("addr:street"),
            "housenumber": tags.get("addr:housenumber"),
            "postcode": tags.get("addr:postcode"),
            "city": tags.get("addr:city")
        })

    return museums


#context für LLM+RAG
def get_osm_context(query):
    place = search_place(query)

    if not place:
        return f"Keine Informationen zu {query} gefunden."

    museums = get_museums_opening_hours(place["lat"], place["lon"])

    texts = []
    for m in museums:
        adress = "".join(filter(None, [m.get("street"), m.get("housenumber"), m.get("postcode"), m.get("city")]))
        texts.append(
            f"Name: {m.get('name')}\n"
            f"Öffnungszeiten: {m.get('opening_hours') or 'Keine Informationen verfügbar'}\n"
            f"Adresse: {adress}\n"
        )
    return "\n".join(texts)

#### 6. Zusammenführung aller Quellen für RAG

In [8]:
# Diese Funktion ist von KI geschrieben
import re

def extract_place_from_query(query: str) -> str:
    """
    Versucht aus einer natürlichen Frage den Ortsnamen oder POI zu extrahieren.
    Z.B. "Welche Öffnungszeiten hat Freilandmuseum Lehde?" -> "Freilandmuseum Lehde"
    """
    # Muster für typische Fragen nach Öffnungszeiten
    patterns = [
        r"welche öffnungszeiten hat (.+?)(?:\?|$)",
        r"wann ist (.+?) geöffnet(?:\?|$)",
        r"ist (.+?) geöffnet(?:\?|$)",
    ]
    query_lower = query.lower()
    for pat in patterns:
        m = re.search(pat, query_lower)
        if m:
            # Den originalen Teil im Query extrahieren (case-insensitive)
            start, end = m.span(1)
            return query[start:end].strip()

    # Fallback: komplette Query nutzen, falls kein Muster passt
    return query.strip()

In [9]:
def build_context_for_RAG(query: str, k: int = 5):

    results = []

    # Wikipedia
    wiki_results = semantic_search_milvus_wiki(query, k=k)
    if any(x in query.lower() for x in ["boot", "paddeln", "kahn"]):
        wiki_results = [] #Keine wiki_results, wenn über Bootsmiete gefragt wird
    for r in wiki_results:
        results.append({
            "text": r["text"],
            "source": r["url"],
            "score": r["score"]
        })

    # Eigene JSON-Daten
    json_results = semantic_search_json(query, k=k)
    for r in json_results:
        results.append({
            "text": r["text"],
            "source": r.get("url", ""),
            "score": r["score"]
        })

    # Bootsverleih (Live Preise). Preise werden nur bei passenden Qieries gegeben
    if any(x in query.lower() for x in ["boot", "paddeln", "kahn"]):
        docs = collect_provider_and_prices()
        boots_text = get_bootsverleih_prices(docs)

        for d in boots_text:
             results.append({
                "text": d["text"],
                "source": d["source"],
                "score": d["score"],
            })

    # OSM Museen
    if any(x in query.lower() for x in ["museum", "öffnung", "geöffnet"]):
        place_query = extract_place_from_query(query)
        osm_context = get_osm_context(place_query)

        if osm_context.strip() and not osm_context.startswith("Keine Informationen"):
            results.append({
                "text": f"[OSM Museen]\n{osm_context}",
                "source": "OpenStreetMap (Nominatim/Overpass)",
                "score": 0.7,
            })

    # Results werden nach Relevanz sortiert
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    selected = results[:k]

    context = "\n\n".join([r["text"] for r in selected])
    sources = list(dict.fromkeys(
        r["source"] for r in results if r.get("source")
    ))

    return context, sources

In [10]:
import ollama

def mistral_ollama(prompt: str, model: str = "mistral"):
    response = ollama.generate(
        model=model,
        prompt=prompt
    )
    return response["response"]

In [11]:
def trim_context_to_fit(context: str, max_chars: int = 12000):
    if len(context) <= max_chars:
        return context
    return context[-max_chars:]

In [12]:
def generate_answer(query: str, k: int = 5):

    context, sources = build_context_for_RAG(query, k=k)
    print("Top‑5 Dokumente:")
    docs_split = context.split("\n\n")[:5]
    for i, doc in enumerate(docs_split):
        src = sources[i] if i < len(sources) else "Keine Quelle verfügbar"
        print(f"Dokument {i+1}:\n{doc}\nQuelle: {src}\n")
    context = trim_context_to_fit(context)
    if not context.strip():
        return {
            "Antwort": "Ich könnte keine passende Informationen finden.",
            "sources": []
        }
    sources = sources[:k]
    prompt = f"""
Du bist ein Reiseassistent für das Region Spreewald.

Beantworte die Frage nur anhand des gegebenen Kontexts.
Wenn du keine Antwort auf die Frage findest, sagt das ehrlich. Du darfst nicht fantasieren oder
nicht existierende Orte, Preise oder Öffnungszeiten nennen. Du darfst keine neuen Quellen erfinden.
URLs sind nur im Abschnitt "Quellen" erlaubt.

Kontext:
{context}

Verfügbare Quellen (du darfst NUR diese verwenden):
{sources}

Frage:
{query}

Formuliere eine klare, strukturierte Antwort auf Deutsch und nenne am Ende die verwendeten Quellen.
"""
    full_prompt = f"<s>[INST] {prompt} [/INST]</s>"
    answer = mistral_ollama(full_prompt, model="mistral")

    return {
        "answer": answer,
        "sources": sources }


In [13]:
# Test 1
query = "Wo kann ich im Spreewald ein Boot mieten?"
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootsverleih Richter und Kajaksport
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter

Dokument 3:

Anbieter: Bootsverleih Gromsch
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gromsch

Dokument 4:

Anbieter: Gasthof zum Slawen - Bootsverleih in Raddusch
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen

Dokument 5:

Anbieter: Bootsverleih Stand up Paddling Spreewald (SuP)
Leistung: Bootsverleih
Preise: - Preis pro Stunde: 10,00 Euro 
- ab sechs Stunden Tagespreis: 50,00 Euro 
- kl

In [14]:
# Test 2
query = "Welche Bootsverleihe gibt es in Lehde?"
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootsverleih Richter und Kajaksport
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter

Dokument 3:

Anbieter: Bootsverleih Gromsch
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gromsch

Dokument 4:

Anbieter: Gasthof zum Slawen - Bootsverleih in Raddusch
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen

Dokument 5:

Anbieter: Bootsvermietung Burkhard Henschelchen in Schlepzig
Leistung: Bootsverleih
Preise: - 1-er Paddelboot (Kajak) - bis 2 Stunden 20,00 Euro - 3 Stunden und Tagesm

In [15]:
# Test 3
query = "Was kostet einen Kahnfahrt für 2 Personen im Spreewald?"
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootsverleih Spreewald-Kanus
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/spreewald-kanus

Dokument 3:

Anbieter: Bootshaus Conrad - Bootsverleih in Burg
Leistung: Bootsverleih
Preise: - 1-Sitzer Paddelboot / Kajak 3 Stunden 20,00 euro | Tagespreis 25,00 Euro 
- 2-Sitzer Paddelboot / Kajak 3 Stunden 30,00 Euro | Tagespreis 35,00 Euro 
- 3-Sitzer Paddelboot / Kajak 3 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 3-Sitzer Kanu / Canadier 2 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 4-Sitzer Kanu / Canadier 3 Stunden 40,00 Euro | Tagespreis 45,00 Euro 
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-conrad

Dokument 4:

Anbieter: Bootsverleih Richt

In [16]:
# Test 4
query = "welche Museen kann ich am Montag besuchen? Mach mir eine Liste"
result = generate_answer(query)
print("Antwort:\n", result["answer"])


Top‑5 Dokumente:
Dokument 1:
Name: Freilandmuseum Lehde
    Ort: Lübbenau, Luebbenau
    Region: Spreewald, Brandenburg
    Öffnungszeiten: täglich 10:00 - 18:00 Uhr
    Eintrittspreise: Erwachsene: 6 €, Kinder unter 18 Jahren: frei
    Zielgruppe: Erwachsene, Jugendliche, Kinder
    Content_type: Museum
    Beschreibung: Das Freilandmuseum Lehde ist ein Freilichtmuseum, das die traditionelle Lebensweise und Kultur der Spreewaldregion zeigt. Besucher können historische Gebäude, Handwerksbetriebe und landwirtschaftliche Einrichtungen besichtigen, um einen Einblick in das Leben der Menschen in der Region zu erhalten. Es gibt auch Veranstaltungen und Workshops, die das kulturelle Erbe des Spreewalds lebendig halten.
Quelle: https://de.wikipedia.org/wiki/Potsdam

Dokument 2:
Kunst, sind unter anderem Werke von Wolf Vostell, Emmett Williams, Christo, Niki de Saint Phalle zu sehen. Auf dem rbb-Gelände in Babelsberg befindet sich ein Standort des Deutschen Rundfunkarchivs (DRA). Am Park Sanss

In [17]:
# Test 4
query = "welche Museen sind am Montag geöffnet? Mach mir eine Liste"
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
Kunst, sind unter anderem Werke von Wolf Vostell, Emmett Williams, Christo, Niki de Saint Phalle zu sehen. Auf dem rbb-Gelände in Babelsberg befindet sich ein Standort des Deutschen Rundfunkarchivs (DRA). Am Park Sanssouci befindet sich das Mühlenmuseum in der Historischen Mühle, mit mühlenkundlicher Ausstellung und praktischer Darstellung des Mahlvorgangs. Kai Desinger öffnete im April 2012 mit der Garage du Pont eine Mischung aus Restaurant und Automuseum. In den Räumen einer ehemaligen Tankstelle sind einige alte Autos ausgestellt, wobei der Schwerpunkt bei französischen Klassikern liegt. Ende 2019 wurde der Betrieb vorübergehend eingestellt. Im Juni 2020 wurde über die Wiedereröffnung berichtet. Theater und Musik Seit 2006 ist das Hans-Otto-Theater in der Schiffbauergasse mit seiner neuen Hauptspielstätte beheimatet. Das Ensemble spielt aber auch im historischen Rokokotheater im Neuen Palais, welches zu den schönsten erhaltenen Theaterräumen des 18. Jah

In [18]:
# Test 5
query = "Welche Öffnungszeiten hat Freilandmuseum Lehde"
results = generate_answer(query)
print("Antwort:\n", results["answer"])

Top‑5 Dokumente:
Dokument 1:
[OSM Museen]
Name: Gurkenmuseum
Öffnungszeiten: Keine Informationen verfügbar
Adresse: An der Dolzke603222Lübbenau/Spreewald
Quelle: OpenStreetMap (Nominatim/Overpass)

Dokument 2:
Name: Heimatstube
Öffnungszeiten: Keine Informationen verfügbar
Adresse: 
Quelle: https://de.wikipedia.org/wiki/Potsdam

Dokument 3:
Name: Haus für Mensch und Natur
Öffnungszeiten: Tu-Su 10:00-17:00
Adresse: Schulstraße903222Lübbenau/Spreewald
Quelle: https://de.wikipedia.org/wiki/Slawenburg_Raddusch

Dokument 4:
Name: Spreewald-Museum Lübbenau
Öffnungszeiten: Apr-Oct: Tu-Su,PH 10:30-18:00; Nov-Mar: Tu-Su,PH 11:00-16:00; Apr-Oct: Mo off; Nov-Mar: Mo off
Adresse: Topfmarkt1203222Lübbenau/Spreewald
Quelle: Keine Quelle verfügbar

Dokument 5:
Name: Trafoturm Raddusch
Öffnungszeiten: 24/7
Adresse: 
Quelle: Keine Quelle verfügbar

Antwort:
  Das Freilandmuseum Lehde ist täglich von April bis September von 10:00 Uhr bis 18:00 Uhr geöffnet [OpenStreetMap (Nominatim/Overpass)].


In [19]:
# Test 6
query = "Wie viel kostet ein Boot für 3 Personen?"
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootshaus Conrad - Bootsverleih in Burg
Leistung: Bootsverleih
Preise: - 1-Sitzer Paddelboot / Kajak 3 Stunden 20,00 euro | Tagespreis 25,00 Euro 
- 2-Sitzer Paddelboot / Kajak 3 Stunden 30,00 Euro | Tagespreis 35,00 Euro 
- 3-Sitzer Paddelboot / Kajak 3 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 3-Sitzer Kanu / Canadier 2 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 4-Sitzer Kanu / Canadier 3 Stunden 40,00 Euro | Tagespreis 45,00 Euro 
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-conrad

Dokument 3:

Anbieter: Bootsverleih Richter und Kajaksport
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter

Dokument 4:

Anbieter: Bootsv

In [20]:
# Test 7
query = "Gibt es Museen, die sich für Kindern nicht geeignet sind?"
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
Name: Plastinarium Museum
    Ort: Guben
    Region: Brandenburg
    Öffnungszeiten: täglich 10:00 - 18:00 Uhr
    Eintrittspreise: Erwachsene: 12 €, Ermäßigt: 10 €, GRUPPENPREISE – AB 10 PERSONEN: Pro Person 10,00 €, Kinder unter 7 Jahren, Lehrer und Referendare (gegen Nachweis: Lehrerausweis oder Bescheinigung der Schule), Begleitperson behinderter Besucher (Kennzeichnung B),Körperspender des IfP, Pressevertreter: frei
    Zielgruppe: Erwachsene, Jugendliche
    Content_type: Museum
    Beschreibung: Das Plastinarium in Guben ist ein Museum, das sich der Plastination von menschlichen Körpern widmet. Besucher können die faszinierenden plastinierten Körper und Organe besichtigen, um mehr über die Anatomie des menschlichen Körpers zu erfahren. Das Museum bietet auch informative Führungen und Ausstellungen, die die Wissenschaft und Kunst der Plastination erklären.
Quelle: https://de.wikipedia.org/wiki/Potsdam

Dokument 2:
Kunst, sind unter anderem Werke von W

In [21]:
# Test 8
query = "Was könnte eine Familie mit 2 Kindern im Burg am Wochenende machen? "
result = generate_answer(query)
print("Antwort:\n", result["answer"])

Top‑5 Dokumente:
Dokument 1:
in der Niederlausitz eine Zeitreise durch 12.000 Jahre Siedlungsgeschichte von der Steinzeit bis zum Mittelalter mit dem slawischen Burgenbau. Im 1000 m² umfassenden Burghof, der als Terrasse des Restaurants sowie als Veranstaltungsort genutzt wird, befinden sich rekonstruierte Speicherbauten und ein Brunnennachbau. In Sichtweite zur Burg verläuft die Bundesautobahn 15. Besucher der Burg verbinden ihren Aufenthalt häufig mit einem Besuch im angrenzenden Spreewald, der über das Dorf Raddusch günstig mit dem Kahn zu erreichen ist. Betreiber der Anlage ist seit August 2024 die Slawenburg gGmbH der Stiftung Slavonic Europe des tschechischen Investors David Chmelík. Vorher gehörte die Burg der Regionalen Entwicklungsgesellschaft Vetschau. Am 4. Dezember 2025 wurde der Pachtvertrag mit Chmelík aufgrund anhaltender Differenzen seitens der Stadt Vetschau gekündigt, am 8. Januar 2026 sollte die Burg wieder der Stadt übergeben werden. Der Pächter hat gegen die Kündig

In [22]:
# Test 9
query = "Was kann man bei schlechtem Wetter im Spreewald unternehmen?"
results = generate_answer(query)
print("Antwort:\n", results["answer"])

Top‑5 Dokumente:
Dokument 1:
Ursache und das Alter der Flusslaufverzweigung im Spreewald sind allerdings immer noch nicht hinreichend geklärt. Böden Im Spreewald herrschen vom Grundwasser beeinflusste Böden (hydromorphe Böden) und Moorböden vor. Auf etwas höher gelegenen, hochwasserfreien Standorten findet man vor allem Gleye. Übergangsformen zur Braunerde sind dabei häufig. Auf den hochwasserbeeinflussten Flächen des östlichen Oberspreewaldes sind Vegen verbreitet, die allerdings meistens Übergänge zu den Gleyböden zeigen. Auf tiefer gelegenen, aber noch nicht vermoorten Flächen kommen Anmoorgleye und Moorgleye vor. Vor allem im westlichen Oberspreewald und im Unterspreewald sind Moore, hier als Niedermoore, weit verbreitet. Sie verzahnen sich über weite Strecken mit den oben erwähnten Gley- und Vegaböden. Fast alle Moorflächen im Spreewald zeigen auf Grund der Grundwasserabsenkung Vererdungserscheinungen. Klima Der Spreewald liegt, wie ganz Brandenburg auch, im Übergangsbereich vom o

#### 7. Evaluation


In [23]:
def generate_evaluation(query: str, k: int=5):
    result = generate_answer(query, k=k)
    answer = result["answer"]

    context, _ = build_context_for_RAG(query, k=k)
    context = trim_context_to_fit(context)



    eval_prompt = """Du bist ein unparteiischer Bewerter eines RAG‑Systems.
Basierend auf der unten angegebenen USER QUERY, dem RETRIEVED CONTEXT und der MODEL ANSWER sollst du die Antwort nach drei Kriterien bewerten:

1. GROUNDEDNESS (1–3)
Wie gut ist die Antwort durch den bereitgestellten Kontext gedeckt?

1 = gut (vollständig gestützt)

2 = mittel (teilweise gestützt)

3 = schlecht (größtenteils nicht gestützt / halluziniert)

2. FAITHFULNESS (1–3)
Wie korrekt und faktentreu ist die Antwort im Vergleich zum Kontext?

1 = gut (keine Widersprüche, keine Halluzinationen)

2 = mittel (kleine Ungenauigkeiten)

3 = schlecht (falsche oder erfundene Inhalte)

3. ANSWER RELEVANCY (1–3)
Wie gut beantwortet die Antwort die Nutzerfrage?

1 = gut (vollständig relevant)

2 = mittel (teilweise relevant)

3 = schlecht (geht am Thema vorbei)

Gib ausschließlich JSON im folgenden Format aus:
{
  "groundedness_score": 1-3,
  "faithfulness_score": 1-3,
  "answer_relevancy_score": 1-3,
  "justification": "kurze Begründung"
}
USER QUERY:
{query}

RETRIEVED CONTEXT:
{context}

MODEL ANSWER:
{answer}

Formuliere eine klare, strukturierte Antwort auf Deutsch
"""
    full_prompt = f"<s>[INST] {eval_prompt} [/INST]</s>"
    judge_response = mistral_ollama(full_prompt, model="mistral")
    try:
        evaluation = json.loads(judge_response)
    except Exception as e:
        evaluation = {
            "groundedness_score": None,
            "faithfulness_score": None,
            "answer_relevancy_score": None,
            "justification": f"JSON-Parsing fehlgeschlagen: {e}",
            "raw_output": judge_response
        }
    return {
        "query": query,
        "answer": answer,
        "evaluation": evaluation
    }

In [24]:
# Evaluationtstest 1
query = "Wo kann ich im Spreewald ein Boot mieten?"
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])

Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootsverleih Richter und Kajaksport
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter

Dokument 3:

Anbieter: Bootsverleih Gromsch
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gromsch

Dokument 4:

Anbieter: Gasthof zum Slawen - Bootsverleih in Raddusch
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen

Dokument 5:

Anbieter: Bootsverleih Stand up Paddling Spreewald (SuP)
Leistung: Bootsverleih
Preise: - Preis pro Stunde: 10,00 Euro 
- ab sechs Stunden Tagespreis: 50,00 Euro 
- kl

In [34]:
# Evaluationtstest 2
query = "Welche Öffnungszeiten hat Freilandmuseum Lehde?"
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])

Top‑5 Dokumente:
Dokument 1:
[OSM Museen]
Name: Gurkenmuseum
Öffnungszeiten: Keine Informationen verfügbar
Adresse: An der Dolzke603222Lübbenau/Spreewald
Quelle: OpenStreetMap (Nominatim/Overpass)

Dokument 2:
Name: Heimatstube
Öffnungszeiten: Keine Informationen verfügbar
Adresse: 
Quelle: https://de.wikipedia.org/wiki/Potsdam

Dokument 3:
Name: Haus für Mensch und Natur
Öffnungszeiten: Tu-Su 10:00-17:00
Adresse: Schulstraße903222Lübbenau/Spreewald
Quelle: https://de.wikipedia.org/wiki/Slawenburg_Raddusch

Dokument 4:
Name: Spreewald-Museum Lübbenau
Öffnungszeiten: Apr-Oct: Tu-Su,PH 10:30-18:00; Nov-Mar: Tu-Su,PH 11:00-16:00; Apr-Oct: Mo off; Nov-Mar: Mo off
Adresse: Topfmarkt1203222Lübbenau/Spreewald
Quelle: Keine Quelle verfügbar

Dokument 5:
Name: Trafoturm Raddusch
Öffnungszeiten: 24/7
Adresse: 
Quelle: Keine Quelle verfügbar

Frage:
 Welche Öffnungszeiten hat Freilandmuseum Lehde?
Antwort:
  Das Freilandmuseum Lehde ist täglich von April bis September von 10:00 Uhr bis 18:00 Uhr 

In [29]:
# Evaluationtstest 3
query = "Was kostet einen Kahnfahrt für 2 Personen im Spreewald?"
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])


Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootshaus Conrad - Bootsverleih in Burg
Leistung: Bootsverleih
Preise: - 1-Sitzer Paddelboot / Kajak 3 Stunden 20,00 euro | Tagespreis 25,00 Euro 
- 2-Sitzer Paddelboot / Kajak 3 Stunden 30,00 Euro | Tagespreis 35,00 Euro 
- 3-Sitzer Paddelboot / Kajak 3 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 3-Sitzer Kanu / Canadier 2 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 4-Sitzer Kanu / Canadier 3 Stunden 40,00 Euro | Tagespreis 45,00 Euro 
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-conrad

Dokument 3:

Anbieter: Bootsverleih Richter und Kajaksport
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter

Dokument 4:

Anbieter: Bootsv

In [30]:
# Evaluationtstest 4
query = "welche Museen kann ich am Montag besuchen? Mach mir eine Liste"
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])

Top‑5 Dokumente:
Dokument 1:
Name: Freilandmuseum Lehde
    Ort: Lübbenau, Luebbenau
    Region: Spreewald, Brandenburg
    Öffnungszeiten: täglich 10:00 - 18:00 Uhr
    Eintrittspreise: Erwachsene: 6 €, Kinder unter 18 Jahren: frei
    Zielgruppe: Erwachsene, Jugendliche, Kinder
    Content_type: Museum
    Beschreibung: Das Freilandmuseum Lehde ist ein Freilichtmuseum, das die traditionelle Lebensweise und Kultur der Spreewaldregion zeigt. Besucher können historische Gebäude, Handwerksbetriebe und landwirtschaftliche Einrichtungen besichtigen, um einen Einblick in das Leben der Menschen in der Region zu erhalten. Es gibt auch Veranstaltungen und Workshops, die das kulturelle Erbe des Spreewalds lebendig halten.
Quelle: https://de.wikipedia.org/wiki/Potsdam

Dokument 2:
Kunst, sind unter anderem Werke von Wolf Vostell, Emmett Williams, Christo, Niki de Saint Phalle zu sehen. Auf dem rbb-Gelände in Babelsberg befindet sich ein Standort des Deutschen Rundfunkarchivs (DRA). Am Park Sanss

In [31]:
# Evaluationtstest 5
query = "Gibt es Museen, die sich für Kindern nicht geeignet sind?"
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])

Top‑5 Dokumente:
Dokument 1:
Name: Plastinarium Museum
    Ort: Guben
    Region: Brandenburg
    Öffnungszeiten: täglich 10:00 - 18:00 Uhr
    Eintrittspreise: Erwachsene: 12 €, Ermäßigt: 10 €, GRUPPENPREISE – AB 10 PERSONEN: Pro Person 10,00 €, Kinder unter 7 Jahren, Lehrer und Referendare (gegen Nachweis: Lehrerausweis oder Bescheinigung der Schule), Begleitperson behinderter Besucher (Kennzeichnung B),Körperspender des IfP, Pressevertreter: frei
    Zielgruppe: Erwachsene, Jugendliche
    Content_type: Museum
    Beschreibung: Das Plastinarium in Guben ist ein Museum, das sich der Plastination von menschlichen Körpern widmet. Besucher können die faszinierenden plastinierten Körper und Organe besichtigen, um mehr über die Anatomie des menschlichen Körpers zu erfahren. Das Museum bietet auch informative Führungen und Ausstellungen, die die Wissenschaft und Kunst der Plastination erklären.
Quelle: https://de.wikipedia.org/wiki/Potsdam

Dokument 2:
Kunst, sind unter anderem Werke von W

In [32]:
# Evaluationtstest 6
query = "Was könnte eine Familie mit 2 Kindern im Burg am Wochenende machen? "
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])

Top‑5 Dokumente:
Dokument 1:
in der Niederlausitz eine Zeitreise durch 12.000 Jahre Siedlungsgeschichte von der Steinzeit bis zum Mittelalter mit dem slawischen Burgenbau. Im 1000 m² umfassenden Burghof, der als Terrasse des Restaurants sowie als Veranstaltungsort genutzt wird, befinden sich rekonstruierte Speicherbauten und ein Brunnennachbau. In Sichtweite zur Burg verläuft die Bundesautobahn 15. Besucher der Burg verbinden ihren Aufenthalt häufig mit einem Besuch im angrenzenden Spreewald, der über das Dorf Raddusch günstig mit dem Kahn zu erreichen ist. Betreiber der Anlage ist seit August 2024 die Slawenburg gGmbH der Stiftung Slavonic Europe des tschechischen Investors David Chmelík. Vorher gehörte die Burg der Regionalen Entwicklungsgesellschaft Vetschau. Am 4. Dezember 2025 wurde der Pachtvertrag mit Chmelík aufgrund anhaltender Differenzen seitens der Stadt Vetschau gekündigt, am 8. Januar 2026 sollte die Burg wieder der Stadt übergeben werden. Der Pächter hat gegen die Kündig

In [33]:
query = "Welche Bootsverleihe gibt es in Lehde? "
evaluation = generate_evaluation(query)

print("Frage:\n", evaluation["query"])
print("Antwort:\n", evaluation["answer"])
print("Evaluation:\n", evaluation["evaluation"])

Top‑5 Dokumente:
Dokument 1:
Anbieter: Bootsverleih am Hafen zur alten Aalreuse
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Dokument 2:

Anbieter: Bootshaus Conrad - Bootsverleih in Burg
Leistung: Bootsverleih
Preise: - 1-Sitzer Paddelboot / Kajak 3 Stunden 20,00 euro | Tagespreis 25,00 Euro 
- 2-Sitzer Paddelboot / Kajak 3 Stunden 30,00 Euro | Tagespreis 35,00 Euro 
- 3-Sitzer Paddelboot / Kajak 3 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 3-Sitzer Kanu / Canadier 2 Stunden 35,00 Euro | Tagespreis 40,00 Euro 
- 4-Sitzer Kanu / Canadier 3 Stunden 40,00 Euro | Tagespreis 45,00 Euro 
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-conrad

Dokument 3:

Anbieter: Bootsverleih Richter und Kajaksport
Leistung: Bootsverleih
Preise: Keine Preise gefunden.
Quelle: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter

Dokument 4:

Anbieter: Bootsv